In [ ]:
# RQ1: CNN Fashion Article Category Classification

import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam

# 1. Create output folders
BASE_DIR = Path("RQ1_CNN_Results")
FIG_DIR = BASE_DIR / "figures"
TABLE_DIR = BASE_DIR / "tables"
TABLE_PNG_DIR = BASE_DIR / "tables_png"

for folder in [FIG_DIR, TABLE_DIR, TABLE_PNG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# 2. Load CSV files
styles_file = "styles.csv"
images_file = "images.csv"

if not Path(styles_file).exists():
    styles_file = "styles(1).csv"

if not Path(images_file).exists():
    images_file = "images(1).csv"

styles = pd.read_csv(styles_file, on_bad_lines="skip")
images = pd.read_csv(images_file)

print("Styles columns:", styles.columns.tolist())
print("Images columns:", images.columns.tolist())

# 3. Prepare filename column
styles["id"] = styles["id"].astype(str)
styles["filename"] = styles["id"] + ".jpg"

if "filename" in images.columns:
    df = pd.merge(styles, images, on="filename", how="inner")
else:
    df = styles.copy()

print("Total rows before image checking:", len(df))

# 4. Set image folder path
possible_image_folders = [
    Path("images"),
    Path("images/images"),
    Path("fashion-dataset/images"),
    Path("fashion-dataset/images/images")
]

IMAGE_FOLDER = None

for folder in possible_image_folders:
    if folder.exists():
        IMAGE_FOLDER = folder
        break

if IMAGE_FOLDER is None:
    raise ValueError("Image folder not found. Keep JPG images inside images or images/images folder.")

print("Using image folder:", IMAGE_FOLDER)
print("Number of JPG files:", len(list(IMAGE_FOLDER.glob("*.jpg"))))

df["image_path"] = df["filename"].apply(lambda x: str(IMAGE_FOLDER / x))
df = df[df["image_path"].apply(os.path.exists)].copy()

print("Total available images after checking:", len(df))

if len(df) == 0:
    raise ValueError("No images found. Check that JPG product images are inside the correct images folder.")

# 5. Select top 5 article categories
df = df.dropna(subset=["articleType"])

top_classes = df["articleType"].value_counts().head(5).index
df = df[df["articleType"].isin(top_classes)].copy()

print("Selected classes:")
print(df["articleType"].value_counts())

# 6. Function to save dataframe as PNG table
def save_table_png(dataframe, title, output_path, figsize=(12, 4)):
    plt.figure(figsize=figsize)
    plt.axis("off")
    table = plt.table(
        cellText=dataframe.values,
        colLabels=dataframe.columns,
        cellLoc="center",
        loc="center"
    )
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 1.4)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()

# 7. Save dataset description table
dataset_table = pd.DataFrame({
    "Description": [
        "Research Question",
        "Total images used",
        "Target column",
        "Number of selected classes",
        "Selected classes",
        "Input image size",
        "Model type"
    ],
    "Value": [
        "RQ1: How accurately can a CNN classify fashion product images into article categories?",
        len(df),
        "articleType",
        len(top_classes),
        ", ".join(top_classes),
        "128 x 128",
        "Custom CNN"
    ]
})

dataset_table.to_csv(TABLE_DIR / "rq1_dataset_description.csv", index=False)
save_table_png(
    dataset_table,
    "RQ1 Dataset Description Table",
    TABLE_PNG_DIR / "rq1_dataset_description.png",
    figsize=(14, 4)
)

# 8. Class distribution figure and table
class_counts = df["articleType"].value_counts()

class_counts_df = class_counts.reset_index()
class_counts_df.columns = ["Article Type", "Number of Images"]

class_counts_df.to_csv(TABLE_DIR / "rq1_class_distribution.csv", index=False)

plt.figure(figsize=(8, 5))
class_counts.plot(kind="bar")
plt.title("Class Distribution of Selected Article Categories")
plt.xlabel("Article Type")
plt.ylabel("Number of Images")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIG_DIR / "rq1_class_distribution.png", dpi=300)
plt.show()

save_table_png(
    class_counts_df,
    "RQ1 Class Distribution Table",
    TABLE_PNG_DIR / "rq1_class_distribution_table.png",
    figsize=(8, 3)
)

# 9. Train validation test split
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["articleType"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["articleType"],
    random_state=42
)

split_table = pd.DataFrame({
    "Split": ["Training", "Validation", "Testing"],
    "Number of Images": [len(train_df), len(val_df), len(test_df)],
    "Percentage": [
        round(len(train_df) / len(df) * 100, 2),
        round(len(val_df) / len(df) * 100, 2),
        round(len(test_df) / len(df) * 100, 2)
    ]
})

split_table.to_csv(TABLE_DIR / "rq1_data_split.csv", index=False)
print(split_table)

save_table_png(
    split_table,
    "RQ1 Data Split Table",
    TABLE_PNG_DIR / "rq1_data_split.png",
    figsize=(7, 3)
)

# 10. Image generators
IMG_SIZE = (128, 128)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_dataframe(
    train_df,
    x_col="image_path",
    y_col="articleType",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True
)

val_generator = test_datagen.flow_from_dataframe(
    val_df,
    x_col="image_path",
    y_col="articleType",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

test_generator = test_datagen.flow_from_dataframe(
    test_df,
    x_col="image_path",
    y_col="articleType",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

class_names = list(train_generator.class_indices.keys())
num_classes = len(class_names)

# 11. Build CNN model
cnn_model = Sequential([
    Conv2D(32, (3, 3), activation="relu", input_shape=(128, 128, 3)),
    BatchNormalization(),
    MaxPooling2D(2, 2),

    Conv2D(64, (3, 3), activation="relu"),
    BatchNormalization(),
    MaxPooling2D(2, 2),

    Conv2D(128, (3, 3), activation="relu"),
    BatchNormalization(),
    MaxPooling2D(2, 2),

    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.5),
    Dense(num_classes, activation="softmax")
])

cnn_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

cnn_model.summary()

# 12. Save model architecture table
architecture_table = pd.DataFrame({
    "Layer": [
        "Conv2D + BatchNorm + MaxPooling",
        "Conv2D + BatchNorm + MaxPooling",
        "Conv2D + BatchNorm + MaxPooling",
        "Flatten",
        "Dense + Dropout",
        "Output Dense Softmax"
    ],
    "Description": [
        "32 filters, 3x3 kernel",
        "64 filters, 3x3 kernel",
        "128 filters, 3x3 kernel",
        "Converts feature maps into vector",
        "128 neurons, ReLU, dropout 0.5",
        f"{num_classes} neurons for article categories"
    ]
})

architecture_table.to_csv(TABLE_DIR / "rq1_cnn_architecture.csv", index=False)

save_table_png(
    architecture_table,
    "RQ1 CNN Model Architecture Table",
    TABLE_PNG_DIR / "rq1_cnn_architecture.png",
    figsize=(10, 3)
)

# 13. Train CNN model
history = cnn_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10
)

# 14. Save accuracy and loss curves
plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.title("CNN Training and Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "rq1_accuracy_curve.png", dpi=300)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("CNN Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "rq1_loss_curve.png", dpi=300)
plt.show()

# 15. Evaluate model
test_generator.reset()

pred_probs = cnn_model.predict(test_generator)
pred_classes = np.argmax(pred_probs, axis=1)
true_classes = test_generator.classes

test_accuracy = accuracy_score(true_classes, pred_classes)

print("Test Accuracy:", test_accuracy)

# 16. Classification report
report = classification_report(
    true_classes,
    pred_classes,
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).transpose()
report_df.to_csv(TABLE_DIR / "rq1_classification_report.csv")

report_png_df = report_df.round(3).reset_index()
report_png_df.rename(columns={"index": "Class"}, inplace=True)

save_table_png(
    report_png_df,
    "RQ1 Classification Results Table",
    TABLE_PNG_DIR / "rq1_classification_report.png",
    figsize=(12, 6)
)

# 17. Confusion matrix
cm = confusion_matrix(true_classes, pred_classes)

cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
cm_df.to_csv(TABLE_DIR / "rq1_confusion_matrix.csv")

plt.figure(figsize=(8, 6))
plt.imshow(cm)
plt.title("CNN Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.xticks(range(len(class_names)), class_names, rotation=45)
plt.yticks(range(len(class_names)), class_names)

for i in range(len(class_names)):
    for j in range(len(class_names)):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.colorbar()
plt.tight_layout()
plt.savefig(FIG_DIR / "rq1_confusion_matrix.png", dpi=300)
plt.show()

# 18. Final accuracy table
accuracy_table = pd.DataFrame({
    "Research Question": [
        "RQ1: How accurately can a CNN classify fashion product images into article categories?"
    ],
    "Model": ["Custom CNN"],
    "Test Accuracy": [round(test_accuracy, 4)],
    "Accuracy Percentage": [str(round(test_accuracy * 100, 2)) + "%"]
})

accuracy_table.to_csv(TABLE_DIR / "rq1_model_accuracy.csv", index=False)

save_table_png(
    accuracy_table,
    "RQ1 Model Accuracy Table",
    TABLE_PNG_DIR / "rq1_model_accuracy.png",
    figsize=(12, 2)
)

# 19. Save model
cnn_model.save(BASE_DIR / "rq1_custom_cnn_model.h5")

print("RQ1 completed successfully.")
print("Figures saved in:", FIG_DIR)
print("CSV tables saved in:", TABLE_DIR)
print("PNG tables saved in:", TABLE_PNG_DIR)